<a href="https://colab.research.google.com/github/aounraza379/AutiSense-AI/blob/main/AutiSense_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q groq gradio openai-whisper gTTS
!apt install ffmpeg -y -q

In [ ]:
import os, time, whisper
import gradio as gr
from gtts import gTTS
from groq import Groq

# --- SETUP ---
GROQ_API_KEY = "API_HERE"
client = Groq(api_key=GROQ_API_KEY)
stt_model = whisper.load_model("tiny")

# --- RAG KNOWLEDGE BASE ---
STRATEGY_KB = {
    "emotional": "Coaching: Validate the feeling first, then suggest a sensory break.",
    "social":    "Coaching: Use 'I feel' statements and explain social cues.",
    "functional":"Coaching: Use 'First/Then' logic and offer specific choices.",
    "crisis":    "Coaching: Stay calm and use very few words."
}

DAILY_TASKS = ["Brush Teeth", "Breakfast", "Lunch", "Playing Game", "Dinner", "Sleep"]
TASK_STEPS  = {
    "Brush Teeth":  ["1. Put paste on brush.", "2. Brush all teeth.", "3. Rinse your mouth."],
    "Breakfast":    ["1. Sit at the table.",   "2. Eat your food.",   "3. Drink your water."],
    "Playing Game": ["1. Pick a toy.",          "2. Play nicely.",     "3. Clean up when done."],
    "Lunch":        ["1. Wash hands.",          "2. Eat your lunch.",  "3. Put your plate away."],
    "Dinner":       ["1. Sit with family.",     "2. Try your food.",   "3. Wipe your face."],
    "Sleep":        ["1. Put on pajamas.",      "2. Get in bed.",      "3. Close your eyes."]
}

# ---------------------------------------------------------------------------
# EMOTION CARDS  –  (button_id, text_sent_to_AI, svg_string)
# ---------------------------------------------------------------------------
def _svg(bg, stroke, eyes, mouth, extra, label):
    return f"""<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 100 120" width="84" height="100">
  <circle cx="50" cy="50" r="36" fill="{bg}" stroke="{stroke}" stroke-width="2.5"/>
  <ellipse cx="14" cy="50" rx="5" ry="8" fill="{bg}" stroke="{stroke}" stroke-width="2"/>
  <ellipse cx="86" cy="50" rx="5" ry="8" fill="{bg}" stroke="{stroke}" stroke-width="2"/>
  {eyes}{mouth}{extra}
  <text x="50" y="108" text-anchor="middle"
        font-family="Arial Rounded MT Bold,Arial,sans-serif"
        font-size="12" font-weight="bold" fill="#222">{label}</text>
</svg>"""

_EO = """<circle cx="34" cy="44" r="6" fill="white" stroke="#333" stroke-width="1.8"/>
  <circle cx="66" cy="44" r="6" fill="white" stroke="#333" stroke-width="1.8"/>
  <circle cx="35" cy="45" r="3" fill="#333"/><circle cx="67" cy="45" r="3" fill="#333"/>
  <circle cx="36" cy="43" r="1" fill="white"/><circle cx="68" cy="43" r="1" fill="white"/>"""

_EW = """<circle cx="34" cy="44" r="8" fill="white" stroke="#333" stroke-width="1.8"/>
  <circle cx="66" cy="44" r="8" fill="white" stroke="#333" stroke-width="1.8"/>
  <circle cx="34" cy="45" r="5" fill="#333"/><circle cx="66" cy="45" r="5" fill="#333"/>
  <circle cx="35" cy="43" r="1.5" fill="white"/><circle cx="67" cy="43" r="1.5" fill="white"/>"""

_ES = """<ellipse cx="34" cy="46" rx="6" ry="3.5" fill="white" stroke="#333" stroke-width="1.8"/>
  <ellipse cx="66" cy="46" rx="6" ry="3.5" fill="white" stroke="#333" stroke-width="1.8"/>
  <path d="M28 42 Q34 38 40 42" stroke="#333" stroke-width="2" fill="none"/>
  <path d="M60 42 Q66 38 72 42" stroke="#333" stroke-width="2" fill="none"/>
  <ellipse cx="34" cy="47" rx="3.5" ry="2" fill="#333"/>
  <ellipse cx="66" cy="47" rx="3.5" ry="2" fill="#333"/>"""

_EA = """<circle cx="34" cy="46" r="6" fill="white" stroke="#333" stroke-width="1.8"/>
  <circle cx="66" cy="46" r="6" fill="white" stroke="#333" stroke-width="1.8"/>
  <circle cx="34" cy="47" r="3" fill="#333"/><circle cx="66" cy="47" r="3" fill="#333"/>
  <line x1="27" y1="37" x2="41" y2="42" stroke="#333" stroke-width="2.5" stroke-linecap="round"/>
  <line x1="73" y1="37" x2="59" y2="42" stroke="#333" stroke-width="2.5" stroke-linecap="round"/>"""

_EX = """<line x1="27" y1="38" x2="41" y2="52" stroke="#333" stroke-width="3" stroke-linecap="round"/>
  <line x1="41" y1="38" x2="27" y2="52" stroke="#333" stroke-width="3" stroke-linecap="round"/>
  <line x1="59" y1="38" x2="73" y2="52" stroke="#333" stroke-width="3" stroke-linecap="round"/>
  <line x1="73" y1="38" x2="59" y2="52" stroke="#333" stroke-width="3" stroke-linecap="round"/>"""

EMOTION_CARDS = [
    ("btn_happy",   "I feel happy",
     _svg("#FFE066","#333",_EO,
          '<path d="M32 60 Q50 74 68 60" stroke="#333" stroke-width="2.5" fill="#FF8A80" stroke-linecap="round"/>',
          '<circle cx="31" cy="58" r="5" fill="#FFAB91" opacity=".7"/><circle cx="69" cy="58" r="5" fill="#FFAB91" opacity=".7"/>',
          "Happy")),

    ("btn_sad",     "I feel sad",
     _svg("#90CAF9","#333",_EO,
          '<path d="M32 66 Q50 54 68 66" stroke="#333" stroke-width="2.5" fill="none" stroke-linecap="round"/>',
          '<line x1="37" y1="53" x2="35" y2="63" stroke="#5599cc" stroke-width="2" opacity=".8"/>'
          '<line x1="63" y1="53" x2="65" y2="63" stroke="#5599cc" stroke-width="2" opacity=".8"/>',
          "Sad")),

    ("btn_angry",   "I feel angry",
     _svg("#EF9A9A","#333",_EA,
          '<path d="M32 66 Q50 56 68 66" stroke="#333" stroke-width="2.5" fill="none" stroke-linecap="round"/>',
          '<path d="M40 28 Q45 22 50 28 Q55 22 60 28" stroke="#c0392b" stroke-width="2" fill="none"/>',
          "Angry")),

    ("btn_scared",  "I feel scared",
     _svg("#CE93D8","#333",_EW,
          '<path d="M34 64 L39 58 L44 64 L49 58 L54 64 L59 58 L64 64" stroke="#333" stroke-width="2" fill="none" stroke-linecap="round"/>',
          "", "Scared")),

    ("btn_tired",   "I am tired",
     _svg("#A5D6A7","#333",_ES,
          '<path d="M35 64 Q50 70 65 64" stroke="#333" stroke-width="2" fill="none" stroke-linecap="round"/>',
          '<path d="M63 28 Q71 18 76 26" stroke="#aaa" stroke-width="1.5" fill="none"/>',
          "Tired")),

    ("btn_excited", "I feel excited",
     _svg("#FFF176","#333",_EO,
          '<ellipse cx="50" cy="63" rx="11" ry="7" fill="#FF8A80" stroke="#333" stroke-width="2"/>',
          '<circle cx="30" cy="57" r="5" fill="#FFAB91" opacity=".7"/><circle cx="70" cy="57" r="5" fill="#FFAB91" opacity=".7"/>',
          "Excited")),

    ("btn_hungry",  "I am hungry",
     _svg("#FFCC80","#333",_EO,
          '<path d="M34 63 Q50 73 66 63" stroke="#333" stroke-width="2.5" fill="#FF8A80" stroke-linecap="round"/>',
          '<text x="50" y="30" text-anchor="middle" font-size="13">🍕</text>',
          "Hungry")),

    ("btn_bathroom","Bathroom",
     _svg("#80DEEA","#333",_EO,
          '<path d="M37 63 Q50 70 63 63" stroke="#333" stroke-width="2" fill="none" stroke-linecap="round"/>',
          '<text x="50" y="30" text-anchor="middle" font-size="13">🚻</text>',
          "Bathroom")),

    ("btn_hurt",    "I am hurt",
     _svg("#FFCDD2","#333",_EX,
          '<path d="M34 66 Q50 58 66 66" stroke="#333" stroke-width="2.5" fill="none" stroke-linecap="round"/>',
          '<path d="M61 26 L66 18 L71 26 L66 22 Z" fill="#E53935"/>',
          "Hurt")),

    ("btn_yes",     "Yes",
     """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 100 120" width="84" height="100">
  <circle cx="50" cy="50" r="36" fill="#A5D6A7" stroke="#2E7D32" stroke-width="3"/>
  <polyline points="27,50 41,64 73,32" stroke="#2E7D32" stroke-width="7"
            fill="none" stroke-linecap="round" stroke-linejoin="round"/>
  <text x="50" y="108" text-anchor="middle"
        font-family="Arial Rounded MT Bold,Arial,sans-serif"
        font-size="12" font-weight="bold" fill="#222">Yes</text>
</svg>"""),

    ("btn_no",      "No",
     """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 100 120" width="84" height="100">
  <circle cx="50" cy="50" r="36" fill="#FFCDD2" stroke="#C62828" stroke-width="3"/>
  <line x1="30" y1="30" x2="70" y2="70" stroke="#C62828" stroke-width="7" stroke-linecap="round"/>
  <line x1="70" y1="30" x2="30" y2="70" stroke="#C62828" stroke-width="7" stroke-linecap="round"/>
  <text x="50" y="108" text-anchor="middle"
        font-family="Arial Rounded MT Bold,Arial,sans-serif"
        font-size="12" font-weight="bold" fill="#222">No</text>
</svg>"""),
]

# ---------------------------------------------------------------------------
# Build the HTML picture board — SVG cards with onclick
# NO <script> here — JS is injected via gr.Blocks(head=...) into <head>
# ---------------------------------------------------------------------------
CARD_CSS = """<style>
.aac-board{display:flex;flex-wrap:wrap;gap:10px;padding:6px 0;}
.aac-card{display:flex;flex-direction:column;align-items:center;
  background:#fff;border:3px solid #ddd;border-radius:14px;
  padding:6px 8px;cursor:pointer;user-select:none;
  transition:border-color .15s,box-shadow .15s,transform .1s;min-width:76px;}
.aac-card:hover{border-color:#42A5F5;box-shadow:0 4px 14px rgba(66,165,245,.35);transform:translateY(-2px);}
.aac-card:active,.aac-card.pressed{transform:scale(.95);border-color:#1565C0;background:#E3F2FD;}
</style>"""

def build_html_board(cards):
    """Build HTML grid of clickable SVG cards."""
    items = ""
    for (eid, _, svg) in cards:
        items += f'<div class="aac-card" onclick="grClick(\'{eid}\', this)">{svg}</div>\n'
    return CARD_CSS + f'<div class="aac-board">{items}</div>'

# ---------------------------------------------------------------------------
# FIX: JavaScript injected via head= parameter into the real <head> tag.
# This is NOT innerHTML — it's a proper <script> that always executes.
# The old code put <script> inside gr.HTML() which uses innerHTML and
# browsers silently refuse to execute scripts added via innerHTML.
# ---------------------------------------------------------------------------
HEAD_JS = """
<script>
function grClick(eid, cardEl) {
    // Find the hidden Gradio button wrapper by elem_id
    var wrapper = document.getElementById(eid);
    if (!wrapper) {
        console.error('grClick: wrapper not found for id:', eid);
        return;
    }
    // Gradio wraps buttons — the real <button> is inside the wrapper div
    var btn = wrapper.querySelector('button');
    if (!btn) {
        // Fallback: maybe the wrapper IS the button
        if (wrapper.tagName === 'BUTTON') { btn = wrapper; }
        else { console.error('grClick: no <button> inside:', eid); return; }
    }
    // Programmatic click — triggers Gradio's event handler
    btn.click();

    // Visual press feedback on the SVG card
    if (cardEl) {
        cardEl.classList.add('pressed');
        setTimeout(function(){ cardEl.classList.remove('pressed'); }, 300);
    }
}
</script>
"""

# ---------------------------------------------------------------------------
# CSS to visually hide the Gradio proxy buttons while keeping them in DOM.
# We use the "sr-only" technique (clip + tiny size) instead of display:none
# because display:none can interfere with event dispatch in some frameworks.
# ---------------------------------------------------------------------------
HIDDEN_BTN_CSS = "\n".join(
    f"#{eid} {{ "
    f"position:absolute !important; width:1px !important; height:1px !important; "
    f"padding:0 !important; margin:-1px !important; overflow:hidden !important; "
    f"clip:rect(0,0,0,0) !important; white-space:nowrap !important; "
    f"border:0 !important; }}"
    for (eid, _, __) in EMOTION_CARDS
)

# ---------------------------------------------------------------------------
# Session
# ---------------------------------------------------------------------------
class Session:
    def __init__(self):
        self.scenario = "parent"
        self.memory   = {}
        self.reset()

    def reset(self):
        self.history = []
        return [], f"Ready! Mode: {self.scenario.upper()}", None

    def update_rag_memory(self, text):
        t = text.lower()
        if "i like" in t:
            self.memory['preference'] = t.split("i like")[-1].strip().replace(".", "")
        if any(w in t for w in ["sad","happy","angry","scared","excited"]):
            self.memory['state'] = "emotional"
        elif any(w in t for w in ["hungry","bathroom","brush","eat","hurt","tired"]):
            self.memory['state'] = "functional"
        else:
            self.memory['state'] = "social"

session = Session()

# --- TTS ---
def generate_speech(text):
    try:
        fname = f"voice_{int(time.time()*1000)}.mp3"
        gTTS(text=text, lang='en').save(fname)
        return fname
    except:
        return None

# --- CORE RAG LOGIC ---
def handle_interaction(text, history):
    if not text: return history, "Ready", None
    session.update_rag_memory(text)
    state      = session.memory.get('state', 'social')
    strategy   = STRATEGY_KB.get(state, STRATEGY_KB["social"])
    preference = session.memory.get('preference', '')

    extra_instruction = ""
    if "hungry" in text.lower():
        pref_msg = f" or {preference}" if preference else ""
        extra_instruction = f" Offer choices: apple, pizza, banana{pref_msg}."

    base_rules = ("- Stay in character "
                  "- Max 1 very short sentence only "
                  "- Use simple words "
                  "- Respond only to what was just said, nothing extra.")
    rag_context  = f"Strategy: {strategy}. {extra_instruction}"
    role_desc    = {"teacher":   "You are Ms. Anna, a teacher. Be structured.",
                    "caretaker": "You are a patient caretaker. Be direct."
                   }.get(session.scenario, "You are a loving parent. Be warm.")

    system_prompt = f"{role_desc}\n{base_rules}\n{rag_context}"
    messages = [{"role":"system","content":system_prompt}]
    for m in history:
        messages.append({"role": m.get("role","user"), "content": m.get("content","")})
    messages.append({"role":"user","content":text})

    try:
        res = client.chat.completions.create(model="llama-3.1-8b-instant", messages=messages)
        ai  = res.choices[0].message.content
    except Exception as e:
        ai = "I am here for you."

    audio = generate_speech(ai)
    new_history = history + [{"role":"user","content":text}, {"role":"assistant","content":ai}]
    return new_history, f"Last heard: {text}", audio

def process_audio(audio, history):
    if not audio: return history, "No audio", None, None
    text = stt_model.transcribe(audio)["text"].strip()
    h, f, a = handle_interaction(text, history)
    return h, f, a, None

# ---------------------------------------------------------------------------
# UI
# ---------------------------------------------------------------------------
with gr.Blocks(
    theme=gr.themes.Soft(),
    head=HEAD_JS,
    css=HIDDEN_BTN_CSS + """
.aac-section-label{font-size:15px;font-weight:700;color:#555;margin:6px 0 2px 2px;}
"""
) as demo:

    gr.Markdown("# 🧠 AutiSense AI: RAG Adaptive Support")

    with gr.Row():
        mode_p = gr.Button("🏠 Parent Mode")
        mode_t = gr.Button("🏫 Teacher Mode")
        mode_c = gr.Button("🤝 Caretaker Mode")

    with gr.Row():
        # ---- LEFT COLUMN ----
        with gr.Column(scale=3):
            chat = gr.Chatbot(type="messages", height=320)

            # SVG picture board (clickable cards)
            gr.HTML("<div class='aac-section-label'>🎭 How do you feel? / What do you need?</div>")
            gr.HTML(build_html_board(EMOTION_CARDS))

            # Hidden Gradio buttons — one per emotion card.
            # The JS grClick() finds these by elem_id and calls .click()
            # which triggers the Gradio event handler below.
            hidden_buttons = []
            for (eid, ai_text, _) in EMOTION_CARDS:
                hb = gr.Button(ai_text, elem_id=eid, visible=True)
                hidden_buttons.append((hb, ai_text))

            mic = gr.Audio(sources=["microphone"], type="filepath", label="🎙️ Record Voice")
            with gr.Row():
                submit_audio_btn = gr.Button("📤 Send & Clear", variant="primary")
                audio_playback   = gr.Audio(autoplay=True, label="AI Voice Output", visible=True)

        # ---- RIGHT COLUMN ----
        with gr.Column(scale=2):
            feedback_label = gr.Label(value="Mode: PARENT")

            gr.Markdown("### 📅 Daily Schedule")
            for t_name in DAILY_TASKS:
                gr.Checkbox(label=t_name, value=False)

            gr.Markdown("---")
            reset_btn = gr.Button("Reset Everything")

    # --- BINDINGS ---
    # Each hidden button → handle_interaction with its emotion text
    for (hb, ai_text) in hidden_buttons:
        hb.click(
            fn=lambda h, t=ai_text: handle_interaction(t, h),
            inputs=[chat],
            outputs=[chat, feedback_label, audio_playback]
        )

    mode_p.click(lambda: (setattr(session,'scenario','parent'),    "Mode: PARENT")[1],    None, feedback_label)
    mode_t.click(lambda: (setattr(session,'scenario','teacher'),   "Mode: TEACHER")[1],   None, feedback_label)
    mode_c.click(lambda: (setattr(session,'scenario','caretaker'), "Mode: CARETAKER")[1], None, feedback_label)

    submit_audio_btn.click(
        fn=process_audio,
        inputs=[mic, chat],
        outputs=[chat, feedback_label, audio_playback, mic]
    )
    reset_btn.click(session.reset, None, [chat, feedback_label, audio_playback])

demo.launch()

In [ ]:
# import os
# os.remove("autisense_memory.pkl")
# print("Memory cleared — fresh start")

In [ ]:
# import os
# print("Memory file exists:", os.path.exists("autisense_memory.pkl"))